# **4.3 Triển khai LLM**

## **4.3.1 Setup — Load model, predict 1 case, lấy top features SHAP**

In [1]:
import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
from dotenv import load_dotenv
load_dotenv()  # đọc file .env, set vào os.environ

True

In [3]:
# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Thêm src vào path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [4]:
# 1. Tìm project_root (chứa folder 'src')
current = Path.cwd().resolve()
project_root = next((p for p in [current, *current.parents] if (p / "src").exists()), None)
if project_root is None:
    raise FileNotFoundError("Không tìm thấy thư mục 'src'")
sys.path.insert(0, str(project_root))
print("project_root =", project_root)

project_root = D:\Python\AIN701_Group_04\demo


In [5]:
# 2. Load model + predict cho 1 hồ sơ vay
from src.models.predict_model import predict_loan, load_model_and_features
from src.models.shap_analysis import get_explainer, get_top_features_for_case

In [6]:
raw_input = {
    'Age': 19, 'Income': 30000, 'LoanAmount': 200000, 'CreditScore': 450,
    'MonthsEmployed': 3, 'NumCreditLines': 5, 'InterestRate': 22.5, 'LoanTerm': 60,
    'DTIRatio': 0.85, 'Education': 'High School', 'EmploymentType': 'Unemployed',
    'MaritalStatus': 'Single', 'HasMortgage': 'No', 'HasDependents': 'No',
    'LoanPurpose': 'Business', 'HasCoSigner': 'No'
}

In [7]:
result, X_input = predict_loan(raw_input)
print(f"Quyết định: {result['status']}  (xác suất = {result['probability_default']:.4f})")

Loaded: XGBClassifier from xgboost.pkl
Features: 24
Quyết định: Rủi ro cao (Default)  (xác suất = 0.9771)


In [8]:
# 3. Tính SHAP cho đúng case này -> lấy top features (phần đang thiếu trong notebook)
model, feature_names = load_model_and_features()
explainer = get_explainer(model)
shap_values = explainer.shap_values(X_input)
top_features = get_top_features_for_case(shap_values, X_input, idx=0, top_n=5)
print(top_features)

Loaded: XGBClassifier from xgboost.pkl
Features: 24
          feature     value  shap_value
0             Age      19.0    0.914149
1    InterestRate      22.5    0.641440
2  MonthsEmployed       3.0    0.438928
3          Income   30000.0    0.429357
4      LoanAmount  200000.0    0.339090


## **4.3.2 Thiết kế Prompt**

In [9]:
from src.models.llm_explainer import explain_loan_decision

explanation_text = explain_loan_decision(result, top_features, raw_input)
print(explanation_text)

Kính gửi Quý khách,

Chúng tôi xin thông báo về kết quả thẩm định hồ sơ vay vốn của Quý khách. Rất tiếc, tại thời điểm hiện tại, hồ sơ của Quý khách **chưa đủ điều kiện để được duyệt**.

**Lý do cho quyết định này:**

*   **Tuổi đời trẻ** (**19 tuổi**) và **thời gian làm việc ngắn** (**3 tháng**) cùng với **tình trạng việc làm hiện tại chưa ổn định**.
*   **Thu nhập hiện tại** (**30.000**) chưa đủ để hỗ trợ khoản **vay lớn** (**200.000**), làm tăng đáng kể rủi ro tài chính.
*   **Lãi suất đề xuất cao** (**22.5%**) phản ánh mức độ rủi ro của hồ sơ theo đánh giá của hệ thống.

**Để cải thiện hồ sơ trong tương lai, Quý khách có thể xem xét các điểm sau:**

*   **Ổn định việc làm** và **tăng cường thu nhập** trong thời gian tới để chứng minh khả năng trả nợ bền vững.
*   **Xây dựng lịch sử tín dụng tốt hơn** bằng cách quản lý hiệu quả các khoản nợ hiện có và duy trì thanh toán đúng hạn.
*   Cân nhắc **giảm số tiền vay** hoặc **tìm người bảo lãnh** khi nộp lại hồ sơ để tăng khả năng được du

## **4.3.3 Gọi LLM API**

In [10]:
!pip install -U google-genai

## **4.3.4 Test với nhiều case khác nhau (Approve / Reject / borderline)**

In [11]:
from src.models.llm_explainer import build_explanation_prompt, generate_explanation

test_cases = {
    "Rủi ro cao": {'Age': 19, 'Income': 30000, 'LoanAmount': 200000, 'CreditScore': 450,
                   'MonthsEmployed': 3, 'NumCreditLines': 5, 'InterestRate': 22.5, 'LoanTerm': 60,
                   'DTIRatio': 0.85, 'Education': 'High School', 'EmploymentType': 'Unemployed',
                   'MaritalStatus': 'Single', 'HasMortgage': 'No', 'HasDependents': 'No',
                   'LoanPurpose': 'Business', 'HasCoSigner': 'No'},
    "An toàn": {'Age': 45, 'Income': 150000, 'LoanAmount': 20000, 'CreditScore': 780,
                'MonthsEmployed': 120, 'NumCreditLines': 2, 'InterestRate': 5.5, 'LoanTerm': 36,
                'DTIRatio': 0.15, 'Education': "Master's", 'EmploymentType': 'Full-time',
                'MaritalStatus': 'Married', 'HasMortgage': 'Yes', 'HasDependents': 'Yes',
                'LoanPurpose': 'Auto', 'HasCoSigner': 'Yes'},
}

for label, ri in test_cases.items():
    r, X_i = predict_loan(ri)
    sv = explainer.shap_values(X_i)
    tf = get_top_features_for_case(sv, X_i, idx=0, top_n=5)
    p = build_explanation_prompt(r, tf, ri)
    explanation = generate_explanation(p)
    print(f"Case: {label} ({r['status']})")
    print(explanation)
    print()

Loaded: XGBClassifier from xgboost.pkl
Features: 24
Case: Rủi ro cao (Rủi ro cao (Default))
Kính gửi Quý khách,

Chúng tôi đã xem xét hồ sơ vay của Quý khách và rất tiếc phải thông báo rằng hiện tại chúng tôi chưa thể phê duyệt khoản vay này.

- **Tại sao lại có quyết định này?**
    - Hồ sơ của Quý khách cho thấy một số yếu tố rủi ro cao, bao gồm độ tuổi **19**, thu nhập **30.000** và thời gian làm việc chỉ **3 tháng**.
    - Số tiền vay mong muốn **200.000** quá lớn so với khả năng tài chính hiện tại, cùng với điểm tín dụng **450** và tình trạng **chưa có việc làm ổn định**.
    - Tỷ lệ nợ trên thu nhập (**0.85**) và mức lãi suất đề xuất (**22.5%**) cũng là những điểm đáng lưu ý.

- **Cách cải thiện hồ sơ của Quý khách:**
    - Tập trung vào việc **tìm kiếm công việc ổn định** và duy trì trong ít nhất **6-12 tháng** để chứng minh khả năng chi trả.
    - Cải thiện **điểm tín dụng** bằng cách thanh toán đúng hạn các khoản nợ hiện có và hạn chế mở thêm các khoản vay mới.
    - Cân nhắc 

## **4.3.5 Nhận xét chất lượng giải thích**